# Route context calibration — 01 source extraction

Single source of truth for the route-context source register, and the
extraction step that turns the loaded ONTD night-train snapshot into an
observation set the calibration can cite.

Run this before `02_route_context_calibration.ipynb`.

Unlike the other three infrastructure domains this one calibrates **no money**:
terrain, timetable buffer, dwell floor and high-speed access are physical and
operational parameters, so there is no currency conversion and no price basis
to escalate anywhere in this package.

In [1]:
# Route context calibration — source extraction
#
# Same contract as the TAC, energy pricing and facility calibrations: notebook
# is truth, CSV is output.

import csv
from pathlib import Path


def _resolve_data_dir() -> Path:
    here = Path.cwd()
    for cand in (
        here / "data",
        here / "backend/models/infrastructure/route_context/calib/data",
    ):
        if cand.parent.exists():
            cand.mkdir(exist_ok=True)
            return cand
    raise RuntimeError(f"cannot locate the calib data directory from {here}")


DATA_DIR = _resolve_data_dir()
SOURCES_DIR = DATA_DIR.parent / "sources"
SOURCES_DIR.mkdir(exist_ok=True)
print(f"data directory:    {DATA_DIR}")
print(f"observation sets:  {SOURCES_DIR}")

REGISTER_REVIEWED = "2026-08-17"

REGISTER_COLUMNS = [
    "source_id",
    "short_id",
    "used",
    "downloaded",
    "title",
    "publisher",
    "pub_year",
    "data_year",
    "kind",
    "url_or_file",
    "date_accessed",
    "reliability_note",
]

register_rows: list[tuple] = []

data directory:    C:\Users\david\PycharmProjects\night-train-target-network-calib\backend\models\infrastructure\route_context\calib\data
observation sets:  C:\Users\david\PycharmProjects\night-train-target-network-calib\backend\models\infrastructure\route_context\calib\sources


## Sources

In [2]:
# --- Sources ---
_R = REGISTER_REVIEWED

register_rows += [
    (
        "RMMS-9",
        "rmms_9",
        "Used",
        "x",
        "Ninth report on monitoring development of the rail market (RMMS), COM(2024)",
        "European Commission (DG MOVE)",
        2024,
        2022,
        "official_report",
        "https://transport.ec.europa.eu/transport-modes/rail/market-monitoring_en",
        _R,
        "Fig.5 and Fig.69 give total train-km and line-km per member state, whose "
        "ratio is the utilisation term; Fig.116 gives punctuality of long-distance "
        "and high-speed passenger services. Covers EU27 plus NO only, and national "
        "punctuality thresholds are NOT harmonised (5 min in some states, 15 in "
        "others) — the single largest weakness in the buffer derivation",
    ),
    (
        "UIC-451-1",
        "uic_451_1",
        "Used",
        "-",
        "UIC Leaflet 451-1: Timetable recovery margins to guarantee timekeeping",
        "International Union of Railways",
        2000,
        None,
        "standard",
        "https://uic.org/",
        _R,
        "The framework the buffer model follows: a regular running supplement plus "
        "a pathing/construction allowance, together in the 3-5 + 3-5 per cent range "
        "that the calibrated 6-10 per cent band sits inside. Used for the shape of "
        "the model and the base term, not for a per-country number",
    ),
    (
        "ONTD-SNAPSHOT",
        "ontd_snapshot",
        "Used",
        "x",
        "Open Night Train Database — active trips, stop sequences and timings",
        "Back-on-Track (Juri Maier et al.)",
        2026,
        2026,
        "database",
        "ontd schema, loaded by db/ontd/loader.py",
        _R,
        "The empirical target: real published night-train timetables. ontd.trip_stop "
        "carries the full intermediate timing chain, so scheduled running time per "
        "leg is measured rather than assumed, and ontd.route_legs pairs it with the "
        "router's own passage time for the same leg. Its limits matter as much as "
        "its coverage: schedules include border control, locomotive changes and "
        "paths deliberately slowed to arrive at a civilised hour, none of which is "
        "buffer",
    ),
    (
        "TN-TOPOGRAPHY",
        "tn_topography",
        "Used",
        "-",
        "Corridor-by-corridor topographic assessment of the target network",
        "Back-on-Track EU (internal, this repository)",
        2026,
        None,
        "internal_assessment",
        "models/infrastructure/route_context/calib/02_route_context_calibration.ipynb",
        _R,
        "Judgement-based cumulative ascent and ruling gradient per country, assigned "
        "by working each country's main night-train corridors against known summit "
        "elevations, valley routings and published ruling gradients. Calibrated to "
        "be right in RANKING and BAND, roughly +/-1 m/km within a band. Not a "
        "measurement and must not be quoted as one — the replacement is a DEM pass "
        "over the routed geometry",
    ),
    (
        "ORR-ROUTER",
        "orr_router",
        "Used",
        "-",
        "OpenRailRouting technical passage times for the ONTD corridors",
        "Geofabrik OpenRailRouting, via models/route/routing/",
        2026,
        2026,
        "model_output",
        "ontd.route_legs.routed_driving_min (db/ontd/projection.py)",
        _R,
        "Constant cruise speed per line class plus a per-stop dynamics surcharge. "
        "Any systematic speed error lands in every country's residual together, "
        "which is why the observation set is read as an upper bound on buffer "
        "rather than as a measurement of it",
    ),
]
print(f"{len(register_rows)} sources")

5 sources


## Write and validate

In [3]:
# --- Write and validate ---


def write_data(name: str, columns: list[str], rows: list[dict]) -> None:
    """STDLIB-ONLY writer, shared by both calib notebooks."""
    path = DATA_DIR / name
    with open(path, "w", newline="", encoding="utf-8") as fh:
        writer = csv.DictWriter(fh, fieldnames=columns, extrasaction="ignore")
        writer.writeheader()
        for row in rows:
            writer.writerow(
                {c: ("" if row.get(c) is None else row[c]) for c in columns}
            )
    print(f"  {name}: {len(rows)} rows")


register_dicts = [dict(zip(REGISTER_COLUMNS, r)) for r in register_rows]
ids = [r["source_id"] for r in register_dicts]
assert len(ids) == len(set(ids)), "duplicate source_id"
assert all(r["url_or_file"] for r in register_dicts), "row with no locator"

write_data("sources_register.csv", REGISTER_COLUMNS, register_dicts)
print(f"register: {len(register_dicts)} sources")

  sources_register.csv: 5 rows
register: 5 sources


## The ONTD buffer observation set

What the real night-train timetable implies about buffer, per leg and per
country. This is the extraction step: it needs a **live database with a loaded
ONTD snapshot**, so it writes an observation set under `sources/` that `02`
then reads like any other source document.

`db/dev/seed.py` never runs this cell — it imports pandas, and the seed
executor skips pandas cells by contract (see `calib/README.md`). That is
deliberate: seeding a fresh container must not depend on a router pass or on
ONTD being loaded.

The residual per leg is

```
implied_buffer_min = scheduled_running_min − routed_driving_min − routed_dynamics_min
```

`scheduled_running_min` is departure(from) → arrival(to), so it excludes dwell
and is directly comparable. It is attributed to countries by each leg's
`country_time_shares`.

In [4]:
# --- ONTD buffer observation set (needs a live DB; skipped by the seed path) ---
# pandas is imported deliberately: it displays the result, AND it is what keeps
# this cell off db/dev/seed.py's execution path.
import json
import os
import statistics
import sys

import pandas as pd

MAX_RESIDUAL_PCT = 300.0
"""Legs padded more than this above routed time are excluded as an operational
stop, a reversal or a different physical path.

Raised from 150 once the distribution was known: the sample centres near +70%,
so a 150% cut was trimming the slow tail — single-track sections, mountain
crossings, freight-path waits — which is signal, not noise. At 300% a leg is
taking four times the router's passage time, which no running-time supplement
explains."""

MIN_LEG_MIN = 10.0
"""Legs shorter than this are excluded. On a 6-minute leg, two minutes of
dispatch and platform release read as a 33% supplement while saying nothing
about running-time margin — short legs would otherwise dominate the percentage
distribution without carrying meaningful minutes."""

MIN_LEGS = 3  # fewer usable legs than this and a country is flagged thin

LEG_COLUMNS = [
    "route_id",
    "trip_id",
    "leg_sequence",
    "from_stop_id",
    "to_stop_id",
    "countries",
    "scheduled_running_min",
    "routed_driving_min",
    "routed_dynamics_min",
    "routed_model_min",
    "implied_buffer_min",
    "implied_buffer_pct",
    "routed_buffer_min",
    "applied_buffer_pct",
    "routed_distance_km",
    "excluded_reason",
]

COUNTRY_COLUMNS = [
    "country_code",
    "n_legs",
    "routed_km",
    "routed_model_min",
    "implied_buffer_min",
    "implied_quota_pct",
    "median_leg_pct",
    "p25_leg_pct",
    "p75_leg_pct",
    "applied_buffer_pct",
]


def _rows(cur) -> list[dict]:
    """Fetch as dicts whatever cursor factory is in play — db/ontd/connection.py
    hands back tuples, db/dev/seed.py uses RealDictCursor."""
    columns = [c.name for c in cur.description]
    fetched = cur.fetchall()
    if fetched and isinstance(fetched[0], dict):
        return [dict(r) for r in fetched]
    return [dict(zip(columns, r)) for r in fetched]


def _shares(raw) -> dict[str, float]:
    if raw is None:
        return {}
    if isinstance(raw, str):
        raw = json.loads(raw)
    # UNK is open water or a ferry leg: no infrastructure manager, no buffer.
    return {cc: float(v) for cc, v in raw.items() if cc != "UNK"}


def _classify(leg: dict) -> tuple[str, float | None]:
    """Why a leg is unusable, or the residual it contributes. Every exclusion is
    named rather than silently dropped — the excluded count is the honest
    measure of how dirty the sample is."""
    sched, drive = leg["scheduled_running_min"], leg["routed_driving_min"]
    dyn = leg["routed_dynamics_min"] or 0
    if drive is None:
        return "routing_failed", None
    if sched is None:
        return "no_timetable", None
    # routed_buffer_min is deliberately NOT part of the model time. The
    # residual is what a real timetable carries above the router's PURE
    # passage time, which is the quantity a buffer quota is supposed to be —
    # subtracting the router's own padding first would measure how well the
    # current quota already fits, not what it should be.
    model = drive + dyn
    if model <= 0:
        return "zero_routed_time", None
    if model < MIN_LEG_MIN:
        return "leg_too_short", None
    pct = (sched - model) / model * 100.0
    if not _shares(leg["country_time_shares"]):
        return "no_country_shares", pct
    if sched < model:
        # The real train is FASTER than the router thinks possible: a router or
        # path problem, never a negative buffer.
        return "router_slower_than_reality", pct
    if pct > MAX_RESIDUAL_PCT:
        return "residual_implausible", pct
    return "", pct


# data/ -> calib/ -> route_context/ -> infrastructure/ -> models/ -> backend/
BACKEND_ROOT = DATA_DIR.parents[4]


def extract_ontd_legs() -> tuple[list[dict], list[dict]]:
    """Read ontd.route_legs and return (per-leg rows, per-country rows)."""
    if str(BACKEND_ROOT) not in sys.path:
        sys.path.insert(0, str(BACKEND_ROOT))
    from db.ontd.connection import connect, resolve_env

    host = resolve_env()["POSTGRES_HOST"]
    print(f"connecting to {host} ...", end=" ")

    with connect() as conn, conn.cursor() as cur:
        cur.execute(
            "SELECT route_id, trip_id, leg_sequence, from_stop_id, to_stop_id, "
            "       scheduled_running_min, routed_driving_min, routed_dynamics_min, "
            "       routed_buffer_min, routed_distance_m, country_time_shares "
            "FROM ontd.route_legs ORDER BY route_id, trip_id, leg_sequence"
        )
        legs = _rows(cur)
    print(f"ok, {len(legs)} legs")

    rows, acc = [], {}
    for leg in legs:
        reason, pct = _classify(leg)
        drive, dyn = leg["routed_driving_min"], leg["routed_dynamics_min"] or 0
        model = None if drive is None else drive + dyn
        residual = (
            None
            if (model is None or leg["scheduled_running_min"] is None)
            else leg["scheduled_running_min"] - model
        )
        shares = _shares(leg["country_time_shares"])
        rows.append(
            {
                "route_id": leg["route_id"],
                "trip_id": leg["trip_id"],
                "leg_sequence": leg["leg_sequence"],
                "from_stop_id": leg["from_stop_id"],
                "to_stop_id": leg["to_stop_id"],
                "countries": "+".join(sorted(shares)),
                "scheduled_running_min": leg["scheduled_running_min"],
                "routed_driving_min": drive,
                "routed_dynamics_min": dyn,
                "routed_model_min": model,
                "implied_buffer_min": residual,
                "implied_buffer_pct": None if pct is None else round(pct, 1),
                "routed_buffer_min": leg["routed_buffer_min"],
                "applied_buffer_pct": (
                    None
                    if (model is None or not model or leg["routed_buffer_min"] is None)
                    else round(leg["routed_buffer_min"] / model * 100.0, 1)
                ),
                "routed_distance_km": (
                    None
                    if leg["routed_distance_m"] is None
                    else round(leg["routed_distance_m"] / 1000.0, 1)
                ),
                "excluded_reason": reason,
            }
        )
        if reason:
            continue
        for cc, share in shares.items():
            a = acc.setdefault(
                cc,
                {
                    "n": 0,
                    "km": 0.0,
                    "model": 0.0,
                    "buffer": 0.0,
                    "pcts": [],
                    "applied": 0.0,
                },
            )
            a["n"] += 1
            a["model"] += model * share
            a["buffer"] += residual * share
            a["pcts"].append(pct)
            a["applied"] += (leg["routed_buffer_min"] or 0.0) * share
            if leg["routed_distance_m"] is not None:
                a["km"] += leg["routed_distance_m"] / 1000.0 * share

    country_rows = []
    for cc, a in sorted(acc.items()):
        pcts = sorted(a["pcts"])
        country_rows.append(
            {
                "country_code": cc,
                "n_legs": a["n"],
                "routed_km": round(a["km"], 1),
                "routed_model_min": round(a["model"], 1),
                "implied_buffer_min": round(a["buffer"], 1),
                "implied_quota_pct": round(a["buffer"] / a["model"] * 100.0, 1),
                "median_leg_pct": round(statistics.median(pcts), 1),
                "p25_leg_pct": round(pcts[len(pcts) // 4], 1),
                "p75_leg_pct": round(pcts[3 * len(pcts) // 4], 1),
                # What the router ALREADY adds, for comparison only. On the
                # 2026-08 snapshot this runs near 39% against quotas of
                # 6-10% — an application bug in the router, visible here
                # because the observation set carries it rather than hiding it.
                "applied_buffer_pct": round(a["applied"] / a["model"] * 100.0, 1),
            }
        )
    return rows, country_rows


def _write_sources(name: str, columns: list[str], rows: list[dict]) -> None:
    with open(SOURCES_DIR / name, "w", newline="", encoding="utf-8") as fh:
        writer = csv.DictWriter(fh, fieldnames=columns, extrasaction="ignore")
        writer.writeheader()
        for row in rows:
            writer.writerow(
                {c: ("" if row.get(c) is None else row[c]) for c in columns}
            )
    print(f"  sources/{name}: {len(rows)} rows")


try:
    leg_rows, country_rows = extract_ontd_legs()
except ModuleNotFoundError as exc:
    # An import failure is a bug here, not a missing database — the backend
    # root is derived from this file's own location and either resolves or
    # does not. Reported separately so it can never be read as "no DB".
    print(
        f"ONTD extraction FAILED to import ({exc}).\n"
        f"  backend root resolved to {BACKEND_ROOT} — it must be the directory "
        "containing db/, models/ and api/. This is a path bug, not a "
        "connection problem."
    )
    leg_rows, country_rows = [], []
except (Exception, SystemExit) as exc:  # noqa: BLE001
    # SystemExit deliberately: db/ontd/connection.py raises it rather than an
    # exception when the database is unreachable, and an uncaught SystemExit
    # inside a notebook kills the kernel instead of skipping the cell.
    print(
        f"ONTD extraction skipped ({type(exc).__name__}: {exc}).\n"
        "  Needs a reachable database with a loaded ONTD snapshot. Outside the "
        "container set POSTGRES_HOST=localhost — the env files name the "
        "compose hostname, which only resolves inside the network. Any "
        "observation set already under sources/ is left untouched, and 02 will "
        "use it — or report the buffer section as awaiting extraction."
    )
    leg_rows, country_rows = [], []

if not leg_rows:
    print("no legs extracted — leaving sources/ as it stands")
elif not country_rows:
    print(
        f"{len(leg_rows)} legs read but none usable — ontd.route_legs has no "
        "routed_* values yet. Rebuild it with db/ontd/projection.py."
    )
    _write_sources("ontd_buffer_legs.csv", LEG_COLUMNS, leg_rows)
else:
    excluded: dict[str, int] = {}
    for r in leg_rows:
        if r["excluded_reason"]:
            excluded[r["excluded_reason"]] = excluded.get(r["excluded_reason"], 0) + 1
    print(f"\n{len(leg_rows)} legs, {len(leg_rows) - sum(excluded.values())} usable")
    for reason, n in sorted(excluded.items(), key=lambda kv: -kv[1]):
        print(f"    excluded {n:5}  {reason}")
    _write_sources("ontd_buffer_legs.csv", LEG_COLUMNS, leg_rows)
    _write_sources("ontd_buffer_by_country.csv", COUNTRY_COLUMNS, country_rows)

pd.DataFrame(country_rows) if country_rows else pd.DataFrame(columns=COUNTRY_COLUMNS)

  note: 'postgres' does not resolve here and we are not in a container — treating it as a compose service name and using localhost instead.
  note: 'openrailrouting' does not resolve here and we are not in a container — treating it as a compose service name and using localhost instead.
  note: OPENRAILROUTING_URL → http://localhost:8989
connecting to localhost ... ok, 1519 legs

1519 legs, 412 usable
    excluded   678  routing_failed
    excluded   303  no_timetable
    excluded    54  router_slower_than_reality
    excluded    52  leg_too_short
    excluded    16  no_country_shares
    excluded     2  zero_routed_time
    excluded     2  residual_implausible
  sources/ontd_buffer_legs.csv: 1519 rows
  sources/ontd_buffer_by_country.csv: 19 rows


,country_code,n_legs,routed_km,routed_model_min,implied_buffer_min,implied_quota_pct,median_leg_pct,p25_leg_pct,p75_leg_pct,applied_buffer_pct
0,AT,56,3600.8,2130.3,674.8,31.7,21.2,9.1,40.9,35.6
1,BE,7,408.7,224.9,115.3,51.3,51.7,33.3,55.8,45.3
2,BG,7,660.5,543.0,332.5,61.2,60.7,30.3,135.9,40.0
3,CH,13,791.4,469.0,236.1,50.3,33.3,17.6,66.7,31.1
4,CZ,24,2451.5,1295.6,579.4,44.7,29.5,18.9,79.3,39.6
5,DE,110,16629.9,7977.2,3891.7,48.8,37.6,18.4,59.7,49.8
6,DK,7,964.2,419.1,247.4,59.0,51.1,45.1,56.7,42.3
7,FR,26,7182.8,3207.7,2513.1,78.3,56.8,38.5,82.0,40.1
8,GB,15,3194.6,1586.0,1103.0,69.5,73.6,48.3,90.7,39.8
9,HR,12,1177.7,1069.8,360.5,33.7,37.9,29.5,46.5,39.6
